# Linke turbidity — exploratory analysis

Exploratory Python workflow for processing the OLUOL Eye2Sky irradiance dataset, retrieving Linke turbidity, comparing it with the pvlib climatology lookup, and checking selected aerosol events with CAMS.

**Data requirement:** place the licensed/authorised `OLUOL.cleaned.nc` file in `data/`. See `data/README.md`. CAMS retrieval cells require a configured Copernicus CDS API account and credentials.

**Reproducibility:** outputs have been cleared. Run cells from top to bottom in an environment containing the packages in `requirements.txt`.


In [ ]:
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pvlib


In [ ]:
# Load the Eye2Sky station dataset from the repository's data directory.
DATA_PATH = Path("data") / "OLUOL.cleaned.nc"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. See data/README.md for access and setup instructions."
    )
ds = xr.open_dataset(DATA_PATH)
ds


In [ ]:
#checking the attributes of needed variables
for var in ["GHI", "DNI", "DHI", "cGHI", "cDNI", "cDHI", "turbidity", "SZA", "SAZ"]:
    print(f"{var}")
    print(ds[var].attrs)
    print()

In [ ]:
#checking the corrdinates
latitude = float(ds["latitude"].values)
longitude = float(ds["longitude"].values)
print(latitude,longitude)

In [ ]:
#looking through different days
day = ds.sel(time="2022-12-12")

plt.figure(figsize=(12,5))
plt.plot(day.time, day["GHI"], label="Measured GHI")
plt.plot(day.time, day["cGHI"], label="Modeled clear-sky GHI (cGHI)", linestyle="--")
plt.xlabel("Time")
plt.ylabel("Irradiance (W/m²)")
plt.title("Measured vs Clear-Sky Modeled GHI  ")
plt.legend()
plt.show()

In [ ]:
#total timestamps available
ds.sizes["time"]

In [ ]:
#filtering out the clearsky moments 
daytime_mask = (ds["SZA"] < 75).values

# Reno-Hansen detection 
ghi_series = ds["GHI"].to_series()
cghi_series = ds["cGHI"].to_series()

rh_mask_raw = pvlib.clearsky.detect_clearsky(
    measured=ghi_series,
    clearsky=cghi_series,
    times=ghi_series.index
)

# Restrict to daytime 
clear_sky_mask = rh_mask_raw.values & daytime_mask
ds["clear_sky_mask"] = ("time", clear_sky_mask)

n_daytime = daytime_mask.sum()
n_clear = clear_sky_mask.sum()
print(f"Daytime timestamps (SZA<75): {n_daytime}")
print(f"Reno-Hansen clear-sky timestamps: {n_clear}  ({n_clear/n_daytime:.2%})")


In [ ]:
# Station elevation metadata check
print("Global elevation attribute:", ds.attrs.get("elevation"), "m")
print("Global height attribute:", ds.attrs.get("height"), "m")
print("Scalar elevation variable:", float(ds["elevation"].values), "m")

# Working interpretation supplied for this project:
# 6 m site elevation above mean sea level + 15 m station height above ground = 21 m.
# The NetCDF metadata does not independently document this breakdown, so 21 m is provisional.


In [ ]:
# Absolute airmass and pressure
# Provisional station altitude above mean sea level: 6 m site elevation + 15 m station height.
# Confirm the vertical reference with the data provider before publishing final results.
altitude = 21.0  # metres above mean sea level

apparent_zenith = ds["SZA"].to_series()

airmass_relative = pvlib.atmosphere.get_relative_airmass(apparent_zenith, model="kastenyoung1989")
pressure = pvlib.atmosphere.alt2pres(altitude)
airmass_absolute = pvlib.atmosphere.get_absolute_airmass(airmass_relative, pressure)

ds["airmass"] = ("time", airmass_absolute.values)

final_mask = ds["clear_sky_mask"]
ds["final_mask"] = final_mask

print(f"Assumed station altitude: {altitude:.1f} m a.s.l.")
print(f"Calculated pressure: {pressure:.2f} Pa")
print(ds["clear_sky_mask"].sum().values, final_mask.sum().values)


In [ ]:
#Calculate Turbidity using the Ineichen Perez Airmass independent formula
doy = ds["time"].dt.dayofyear.values
I0 = 1367 * (1 + 0.033 * np.cos(2 * np.pi * doy / 365))

fh1 = np.exp(-altitude / 8000.0)
b = 0.664 + 0.163 / fh1

DNI = ds["DNI"].values
AM = ds["airmass"].values
mask = ds["final_mask"].values

TL_raw = np.full(len(DNI), np.nan)
valid = mask & (DNI > 0)
TL_raw[valid] = 1 + np.log(b * I0[valid] / DNI[valid]) / (0.09 * AM[valid])

TL_derived = TL_raw.copy()
low_tl = valid & (TL_raw < 2)
TL_derived[low_tl] = TL_raw[low_tl] - 0.25 * np.sqrt(2 - TL_raw[low_tl])

ds["TL_derived"] = ("time", TL_derived)

print(valid.sum())
print(f"Points receiving low-T_L correction: {low_tl.sum()}")
print(np.nanmean(TL_derived), np.nanmedian(TL_derived))
print(np.nanmin(TL_derived), np.nanmax(TL_derived))

In [ ]:
#Plotting the given Turbidity values
import matplotlib.dates as mdates

valid = ds["final_mask"].values

fig, ax = plt.subplots(figsize=(15,10))
ax.scatter(ds["time"].values[valid], ds["TL_derived"].values[valid], s=8, color="tab:blue")

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax.set_xlabel("Date")
ax.set_ylabel("Self-computed Linke turbidity (T_L)")
ax.set_title("Self-computed Linke Turbidity — OLUOL, April 2022 to March 2023")
plt.tight_layout()
plt.show()


In [ ]:
#Checking through the Turbidity statistics
df = ds[["TL_derived"]].to_dataframe().dropna()
print(df["TL_derived"].describe())
print(df["TL_derived"].idxmax(), df["TL_derived"].max())

#Turbidity count size per month
monthly_stats = df.groupby(df.index.to_period("M"))["TL_derived"].describe()
print(monthly_stats)


In [ ]:
#checking the peak turbidity 
window = ds["TL_derived"].sel(time=slice("2022-06-18 08:00", "2022-06-18 12:00")).to_series().dropna()
print(window)
print(f"\nNumber of points: {len(window)}")

In [ ]:
window = ds["TL_derived"].sel(time=slice("2022-06-18 07:00", "2022-06-18 12:00")).to_series()

half_hourly = window.resample("30min").mean()
print(half_hourly)
half_hourly_clean = half_hourly.dropna()
print(half_hourly_clean)

In [ ]:
#getting the Climatological turbidity value
climatology = pvlib.clearsky.lookup_linke_turbidity(
    time=ds["time"].to_index(),
    latitude=float(ds["latitude"].values),
    longitude=float(ds["longitude"].values)
)

ds["TL_climatology"] = ("time", climatology.values)

print(ds["TL_climatology"].to_series().resample("MS").mean())

In [ ]:
#comparing the climatological turbidity with median of calculated turbidity
derived = ds["TL_derived"].to_series().dropna()
climatology_series = ds["TL_climatology"].to_series()

comparison = pd.DataFrame({
    "TL_derived_median": derived.groupby(derived.index.to_period("M")).median(),
    "TL_derived_count": derived.groupby(derived.index.to_period("M")).count(),
    "TL_climatology_mean": climatology_series.groupby(climatology_series.index.to_period("M")).mean()
})

comparison["difference"] = comparison["TL_climatology_mean"] - comparison["TL_derived_median"]
comparison["pct_difference"] = 100 * comparison["difference"] / comparison["TL_derived_median"]

print(comparison)

In [ ]:
#plotting both 
if not isinstance(comparison.index, pd.DatetimeIndex):
    comparison.index = comparison.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12,6))
ax.plot(comparison.index, comparison["TL_derived_median"], marker="o", label="Self-computed (median)")
ax.plot(comparison.index, comparison["TL_climatology_mean"], marker="o", label="pvlib climatology")
ax.set_ylabel("Linke turbidity")
ax.set_title("Measured T_L vs Climatological T_L by month — OLUOL")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2 — Monthly relative difference between climatology and station-derived median
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
values = comparison["pct_difference"]
bars = ax.bar(comparison.index.strftime("%b %Y"), values, edgecolor="black", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Relative difference (%)")
ax.set_xlabel("Month")
ax.set_title("Monthly Relative Difference: pvlib Climatology vs Station-Derived $T_L$")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
for bar, value in zip(bars, values):
    ax.annotate(f"{value:.1f}%", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig("figures/fig02_monthly_relative_difference.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig02_monthly_relative_difference.png")

In [ ]:
# Figure 3 — Monthly climatology vs station-derived Linke turbidity
x = comparison["TL_derived_median"]
y = comparison["TL_climatology_mean"]
valid = x.notna() & y.notna()
xv, yv = x[valid], y[valid]

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(xv, yv, s=55, alpha=0.85, edgecolor="black", linewidth=0.5)
lower = min(float(xv.min()), float(yv.min()))
upper = max(float(xv.max()), float(yv.max()))
padding = max((upper - lower) * 0.05, 0.05)
limits = [lower - padding, upper + padding]
ax.plot(limits, limits, linestyle="--", color="black", label="1:1 line")
for date, xval, yval in zip(xv.index, xv, yv):
    ax.annotate(date.strftime("%b"), (xval, yval), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Station-derived monthly median $T_L$")
ax.set_ylabel("pvlib climatological monthly mean $T_L$")
ax.set_title("Monthly Climatology vs Station-Derived Linke Turbidity")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("figures/fig03_monthly_turbidity_scatter.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig03_monthly_turbidity_scatter.png")

In [ ]:
lowest_month = comparison["TL_climatology_mean"].idxmin()
highest_month = comparison["TL_climatology_mean"].idxmax()

print("Lowest climatological turbidity month:", lowest_month, comparison.loc[lowest_month, "TL_climatology_mean"])
print("Highest climatological turbidity month:", highest_month, comparison.loc[highest_month, "TL_climatology_mean"])

In [ ]:
#effect on solar irradiance
from pvlib import solarposition, irradiance, clearsky

def clearsky_for_month(period, day_of_month=15):
    day = pd.Timestamp(period.year, period.month, day_of_month)
    times = pd.date_range(day, day + pd.Timedelta(hours=23, minutes=59), freq="1min", tz="UTC")
    solpos = solarposition.get_solarposition(times, latitude, longitude)
    apparent_zenith = solpos["apparent_zenith"]
    airmass_rel = pvlib.atmosphere.get_relative_airmass(apparent_zenith)
    airmass_abs = pvlib.atmosphere.get_absolute_airmass(airmass_rel, pressure)
    dni_extra = irradiance.get_extra_radiation(times)

    tl_clim = comparison.loc[period, "TL_climatology_mean"]
    tl_meas = comparison.loc[period, "TL_derived_median"]

    cs_clim = clearsky.ineichen(apparent_zenith, airmass_abs, tl_clim, altitude, dni_extra)
    cs_meas = clearsky.ineichen(apparent_zenith, airmass_abs, tl_meas, altitude, dni_extra)
    return times, cs_clim, cs_meas, tl_clim, tl_meas

def annotate_peaks(ax, cs):
    text = (f"max GHI: {cs['ghi'].max():.0f} W/m²\n"
            f"max DNI: {cs['dni'].max():.0f} W/m²\n"
            f"max DHI: {cs['dhi'].max():.0f} W/m²")
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=9, va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)

for col, period in enumerate([lowest_month, highest_month]):
    times, cs_clim, cs_meas, tl_clim, tl_meas = clearsky_for_month(period)

    axes[0, col].plot(times, cs_clim["ghi"], label="GHI")
    axes[0, col].plot(times, cs_clim["dni"], label="DNI")
    axes[0, col].plot(times, cs_clim["dhi"], label="DHI")
    axes[0, col].set_title(f"{period} — climatology T_L={tl_clim:.2f}")
    annotate_peaks(axes[0, col], cs_clim)

    axes[1, col].plot(times, cs_meas["ghi"], label="GHI")
    axes[1, col].plot(times, cs_meas["dni"], label="DNI")
    axes[1, col].plot(times, cs_meas["dhi"], label="DHI")
    axes[1, col].set_title(f"{period} — self-computed T_L={tl_meas:.2f}")
    annotate_peaks(axes[1, col], cs_meas)

for ax in axes.flat:
    ax.legend(loc="upper right")
axes[0,0].set_ylabel("Irradiance (W/m²)")
axes[1,0].set_ylabel("Irradiance (W/m²)")

fig.suptitle("Clear-Sky Model: Lowest vs Highest Turbidity Month, Climatology vs Self-Computed")
plt.tight_layout()
plt.show()

In [ ]:
#June 18 
day = ds.sel(time="2022-06-18")

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(day.time, day["GHI"], label="Measured GHI")
ax.plot(day.time, day["cGHI"], label="Modeled clear-sky GHI", linestyle="--")
ax.plot(day.time, day["DNI"], label="Measured DNI")
ax.plot(day.time, day["cDNI"], label="Modeled clear-sky DNI", linestyle="--")
ax.axvspan(pd.Timestamp("2022-06-18 07:00"), pd.Timestamp("2022-06-18 11:05"),
           color="red", alpha=0.15, label="Outlier window")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Irradiance (W/m²)")
ax.set_title("June 18, 2022 — Measured vs Modeled Irradiance")
ax.legend()
plt.show()

In [ ]:
df = ds[["TL_derived"]].to_dataframe().dropna()
peak_time = df["TL_derived"].idxmax()
print("Peak timestamp:", peak_time)

check_times = pd.to_datetime([peak_time])
print(ds["SZA"].sel(time=check_times, method="nearest").values)
print(ds["airmass"].sel(time=check_times, method="nearest").values)

In [ ]:
#Getting the AOD from CAMS
import cdsapi

client = cdsapi.Client()
client.retrieve(
    "cams-global-reanalysis-eac4",
    {
        "date": "2022-06-18/2022-06-18",
        "time": ["06:00", "09:00", "12:00", "15:00", "18:00"],
        "variable": "total_aerosol_optical_depth_550nm",
        "area": [54, 7, 52, 9],
        "format": "netcdf",
    },
    "cams_aod_20220618_3hr.nc"
)

print("Download complete.")

In [ ]:
aod_ds_day = xr.open_dataset("cams_aod_20220618_3hr.nc")

aod_point_day = aod_ds_day["aod550"].sel(
    latitude=53.15348, longitude=8.16192, method="nearest"
)

for t, v in zip(aod_point_day["valid_time"].values, aod_point_day.values):
    print(pd.Timestamp(t), "->", round(float(v), 4))

In [ ]:
client.retrieve(
    "cams-global-reanalysis-eac4",
    {
        "date": "2022-06-18/2022-06-18",
        "time": ["06:00", "09:00", "12:00", "15:00", "18:00"],
        "variable": [
            "dust_aerosol_optical_depth_550nm",
            "organic_matter_aerosol_optical_depth_550nm",
            "black_carbon_aerosol_optical_depth_550nm",
            "sulphate_aerosol_optical_depth_550nm",
            "sea_salt_aerosol_optical_depth_550nm",
            "total_column_water_vapour"
        ],
        "area": [54, 7, 52, 9],
        "format": "netcdf",
    },
    "cams_species_20220618_v3.nc"
)

print("Download complete.")

In [ ]:
#Species wise data from CAM
species_ds = xr.open_dataset("cams_species_20220618_v3.nc")

species_point = species_ds.sel(latitude=53.15348, longitude=8.16192, method="nearest")

species_df = pd.DataFrame({
    "dust": species_point["duaod550"].values,
    "organic_matter": species_point["omaod550"].values,
    "black_carbon": species_point["bcaod550"].values,
    "sulphate": species_point["suaod550"].values,
    "sea_salt": species_point["ssaod550"].values,
    "water_vapour_kg_m2": species_point["tcwv"].values,
}, index=species_point["valid_time"].values)

print(species_df)

In [ ]:
# Figure 4 — CAMS aerosol-species AOD time series (18 June 2022)
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

species_to_plot = species_df.drop(columns=["water_vapour_kg_m2"], errors="ignore")
species_to_plot = species_to_plot.apply(pd.to_numeric, errors="coerce")

fig, ax = plt.subplots(figsize=(9, 5))
for column in species_to_plot.columns:
    ax.plot(species_to_plot.index, species_to_plot[column],
            marker="o", linewidth=1.8, label=column.replace("_", " ").title())

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Aerosol optical depth at 550 nm")
ax.set_title("CAMS Aerosol-Species AOD — OLUOL, 18 June 2022")
ax.grid(alpha=0.25)
ax.legend(ncol=2, frameon=False)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig("figures/fig04_cams_aerosol_species.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: figures/fig04_cams_aerosol_species.png")